# Bounded Forex RL pipeline

This notebook runs one **synthetic daily OHLCV** walk-forward fold with a deliberately small, actually trained DQN. It is a wiring check, not evidence of a profitable FX strategy. Run all cells after installing `.[research]`. Full historical training remains opt-in outside this demo.


In [1]:
from dataclasses import replace
from time import perf_counter
import numpy as np
import pandas as pd
from stable_baselines3 import DQN
from forex_rl import ExperimentConfig, compute_indicators, build_walkforward_folds, make_scaled_env, backtest_model, run_baselines, seed_everything
from forex_rl.walk_forward import slice_fold

started = perf_counter()
config = replace(ExperimentConfig(), symbol='SYNTHETIC_OHLCV', start_date='2020-01-01',
                 end_date='2020-06-29', random_seed=7, window_size=4, min_train_months=2,
                 val_months=1, test_months=1, step_months=1, n_envs=1, wf_n_envs=1,
                 total_timesteps=96, eval_freq=0, max_bars_in_trade=3,
                 transaction_cost=0.0001, slippage=0.00005)
seed_everything(config.random_seed)
feature_cols = ['ema20', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower', 'atr_n', 'ret']
price_cols = ['open', 'high', 'low', 'close', 'atr']
print('Source: deterministic synthetic daily OHLCV; seed:', config.random_seed)
print('Training steps:', config.total_timesteps, 'DQN learning starts: 8; CPU')
print('Exact experiment config:', config.to_dict())


Source: deterministic synthetic daily OHLCV; seed: 7
Training steps: 96 DQN learning starts: 8; CPU
Exact experiment config: {'symbol': 'SYNTHETIC_OHLCV', 'start_date': '2020-01-01', 'end_date': '2020-06-29', 'window_size': 4, 'position_risk_frac': 0.01, 'sl_atr_mult': 1.8, 'tp_atr_mult': 2.2, 'max_bars_in_trade': 3, 'transaction_cost': 0.0001, 'slippage': 5e-05, 'invalid_action_penalty': 0.0, 'dd_penalty': 0.0, 'turnover_penalty': 0.0, 'min_train_months': 2, 'val_months': 1, 'test_months': 1, 'step_months': 1, 'use_rolling': True, 'rolling_train_months': 36, 'random_seed': 7, 'algo': 'DQN', 'total_timesteps': 96, 'eval_freq': 0, 'n_envs': 1, 'wf_n_envs': 1, 'rppo_n_steps': 128, 'qrdqn_n_quantiles': 51, 'min_trades_val': 4, 'log_dir': './logs_rl', 'report_dir': './logs_rl/reports'}


In [2]:
dates = pd.date_range('2020-01-01', periods=180, freq='D')
rng = np.random.default_rng(config.random_seed)
close = 1.10 + np.cumsum(rng.normal(0, 0.001, len(dates)))
raw = pd.DataFrame({'open': close - 0.0002, 'high': close + 0.001,
                    'low': close - 0.001, 'close': close, 'volume': 1000.0}, index=dates)
data = compute_indicators(raw)
fold = build_walkforward_folds(data, config.min_train_months, config.val_months,
                               config.test_months, config.step_months)[0]
train, validation, test = (slice_fold(data, fold, part) for part in ('train', 'val', 'test'))
assert train.index.max() < validation.index.min() < test.index.min()
print('Raw/prepared rows:', len(raw), len(data))
print('Train/validation/test rows:', len(train), len(validation), len(test))
print('Fold boundaries (end exclusive):', fold)


Raw/prepared rows: 180 161
Train/validation/test rows: 60 31 30
Fold boundaries (end exclusive): {'fold': 1, 'train_start': Timestamp('2020-01-20 00:00:00'), 'train_end': Timestamp('2020-03-20 00:00:00'), 'val_start': Timestamp('2020-03-20 00:00:00'), 'val_end': Timestamp('2020-04-20 00:00:00'), 'test_start': Timestamp('2020-04-20 00:00:00'), 'test_end': Timestamp('2020-05-20 00:00:00')}


In [3]:
train_env, validation_env, scaler = make_scaled_env(
    train, validation, feature_cols, price_cols, config.environment_kwargs,
    config.random_seed, n_envs=1)
assert scaler.means.equals(train[feature_cols].mean())
model = DQN('MlpPolicy', train_env, seed=config.random_seed, device='cpu', verbose=0,
            policy_kwargs={'net_arch': [32, 32]}, learning_rate=0.001,
            buffer_size=256, learning_starts=8, batch_size=8, train_freq=1,
            gradient_steps=1)
try:
    model.learn(total_timesteps=config.total_timesteps)
finally:
    train_env.close()
    validation_env.close()
print('Actual model timesteps:', model.num_timesteps)


Actual model timesteps: 96


In [4]:
validation_result = backtest_model(model, scaler, validation, feature_cols, price_cols, config)
agent = backtest_model(model, scaler, test, feature_cols, price_cols, config)
baselines = run_baselines(test, config)
rows = {'DQN test': agent, **{name + ' test': value for name, value in baselines.items()}}
summary = pd.DataFrame({name: {'trades': result['metrics']['trade_count'],
                               'total_R': result['metrics']['total_R'],
                               'sharpe': result['metrics']['sharpe'],
                               'sortino': result['metrics']['sortino'],
                               'maxDD_pct': result['metrics']['maxDD_pct'],
                               'turnover': result['metrics']['turnover']}
                        for name, result in rows.items()}).T
print('Validation (model selection was not performed):', validation_result['metrics'])
print('Held-out test and matched baselines:')
print(summary.to_string())
print('Elapsed seconds:', round(perf_counter() - started, 2))


Validation (model selection was not performed): {'trades': 7, 'win_rate': 0.7142857142857143, 'profit_factor': 1.3591724684649957, 'total_R': 0.33390749721984175, 'avg_R': 0.047701071031405966, 'maxDD_R': -0.5502065802251502, 'MAR_R': 0.6068765973013325, 'sharpe': 1.0690051466559356, 'sortino': 1.9730209952577291, 'maxDD_pct': -0.007085927777211953, 'CAGR': 0.027103224972424567, 'calmar': 3.824936666668746, 'turnover': 39.797789548726215, 'trade_count': 7}
Held-out test and matched baselines:
             trades   total_R    sharpe   sortino  maxDD_pct   turnover
DQN test        7.0  0.236877  0.523594  0.800806  -0.010993  39.435200
flat test       0.0  0.000000       NaN       NaN   0.000000   0.000000
random test     1.0 -0.192376 -5.513230 -5.287307  -0.001924   5.419552
trend test      7.0  0.949327  2.503910  5.003549  -0.005322  39.521955
Elapsed seconds: 1.67


## Interpretation

The seeded synthetic series has no claim to real EUR/USD behavior. This single fold and 96 training steps verify interfaces and reporting only. No full historical walk-forward run, large-model training, hyperparameter search, or robustness analysis was run. Sharpe/Sortino can be undefined when the tiny sample has no return variation.
